# Data analysis, session 1B: Visualize, diagnose, clean
**October 8, 10:35-noon (85 minutes).** Start from a fresh kernel; this notebook does not depend on session 1A's variables.

| Time | Activity |
|---|---|
| 10:35-11:10 | Choose charts; pandas + matplotlib; returns and rolling windows |
| 11:10-11:40 | Missing values, text, duplicates, invalid data |
| 11:40-11:55 | Guided cleaning exercise and validation |
| 11:55-noon | Explain a decision; preview SQL and homework |

Our question: *Which observations deserve investigation before we compare these stocks?*
The examples use the same synthetic OHLCV setting as the original visualization and cleaning notebooks.
Data generation is supplied: read its contract, run it, and spend class time on analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
%matplotlib inline
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
def make_prices(seed=42, days=120):
    """Reproducible teaching data, not observed market prices or an exchange calendar."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range('2024-01-02', periods=days)
    frames = []
    for ticker, initial, daily_vol in [('TECH', 150, 0.02), ('BANK', 45, 0.01), ('ENERGY', 80, 0.025)]:
        close = initial * np.exp(np.cumsum(rng.normal(0.0004, daily_vol, days)))
        opening = close * np.exp(rng.normal(0, 0.003, days))
        frames.append(pd.DataFrame({
            'date': dates, 'ticker': ticker, 'open': opening,
            'high': np.maximum(opening, close) * 1.005,
            'low': np.minimum(opening, close) * 0.995, 'close': close,
            'volume': rng.integers(1_000_000, 5_000_000, days).astype(float)}))
    return pd.concat(frames, ignore_index=True)
prices = make_prices()
prices.head()

## 1. Match the chart to the question
Line: change over time. Histogram/KDE: distribution. Scatter: relationship. Bar: categories. Box: distribution across groups.
Compute returns within each ticker after sorting. `pct_change` gives a fraction: multiply by 100 only when displaying percent.
The first return for each ticker is undefined; do not silently fill it.

In [ ]:
prices = prices.sort_values(['ticker', 'date'])
prices['return'] = prices.groupby('ticker')['close'].pct_change(fill_method=None)
wide = prices.pivot(index='date', columns='ticker', values='close')
returns = wide.pct_change(fill_method=None)
normalized = wide / wide.iloc[0] * 100
normalized.plot(title='Synthetic prices, normalized to 100', figsize=(10, 4), ylabel='Indexed price')
plt.show()

### One Figure, several Axes
`plt.subplots` creates the canvas and chart objects. Pass `ax=` to pandas plotting so you control where a plot appears.
Before running, choose a chart for each question below.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout='constrained')
(returns['TECH'] * 100).plot.hist(ax=axes[0, 0], bins=25, title='TECH return distribution')
axes[0, 0].set_xlabel('Daily return (%)')
(returns * 100).plot.scatter(x='TECH', y='BANK', ax=axes[0, 1], alpha=0.5,
                           title='Do TECH and BANK move together?')
axes[0, 1].set(xlabel='TECH return (%)', ylabel='BANK return (%)')
(prices.groupby('ticker')['volume'].mean() / 1e6).plot.bar(
    ax=axes[1, 0], rot=0, title='Mean daily volume', ylabel='Million shares')
(returns * 100).plot.box(ax=axes[1, 1], title='Return spread by stock', ylabel='Daily return (%)')
plt.show()
print('Return correlations:')
display(returns.corr().round(3))

### Rolling statistics and saving a figure
A 20-observation window needs 20 observations. The empty warm-up region is expected.
We use 252 as an annualization convention; the synthetic business-day index is not an exchange holiday calendar.
The advanced scatter-matrix, lag/ACF, KDE, and plotting-style comparisons remain in [visualization reference](reference/4_visualization_seeing_your_data.ipynb).

In [ ]:
tech = prices.loc[prices['ticker'] == 'TECH'].set_index('date')
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, layout='constrained')
tech['close'].plot(ax=axes[0], label='Close')
tech['close'].rolling(20).mean().plot(ax=axes[0], label='20-day mean')
axes[0].set(ylabel='Price', title='TECH: trend and variability'); axes[0].legend()
(tech['return'].rolling(20).std() * np.sqrt(252) * 100).plot(ax=axes[1])
axes[1].set(ylabel='Annualized volatility (%)', xlabel='Date')
output_dir = course_root / 'Lectures/Lecture 2/outputs'
output_dir.mkdir(exist_ok=True)
fig.savefig(output_dir / 'tech_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Make a diagnosis before choosing a repair
`None` and `NaN` are missing; infinity is non-finite but **not** counted by `isna()`.
There is no universal rule that “less than 5% missing” is safe to drop. Ask why values are missing.
We will inject a few known faults, then repair those faults without changing valid extremes.

In [ ]:
raw = make_prices()
raw.loc[2, 'ticker'] = ' <b>tech</b> '
raw.loc[[4, 5, 120], 'close'] = np.nan  # includes BANK's first observation
raw.loc[10, 'close'] *= 100
raw.loc[12, 'volume'] = -100
raw.loc[14, 'volume'] = np.inf
raw = pd.concat([raw, raw.iloc[[3]]], ignore_index=True).sample(frac=1, random_state=7)
raw.info()
display(raw.isna().sum())
print('Non-finite volume:', (~np.isfinite(raw['volume'])).sum())
print('Ticker variants:', raw['ticker'].unique())
raw.plot.scatter(x='date', y='close', title='Raw prices: investigate the spike', figsize=(10, 3))
plt.show()

### Plan: identity, order, domain checks, repair, validation
Normalize ticker names **before** grouping or detecting duplicate `(ticker, date)` keys.
For this exercise the repeated observation is an exact duplicate. Conflicting vendor records require a documented source rule, not arbitrary averaging.

In [ ]:
clean = raw.copy()
clean['ticker'] = clean['ticker'].str.replace(r'<[^>]+>', '', regex=True).str.strip().str.upper()
key = ['ticker', 'date']
display(clean.loc[clean.duplicated(key, keep=False)].sort_values(key))
clean = clean.drop_duplicates().sort_values(key).reset_index(drop=True)
assert not clean.duplicated(key).any(), 'Resolve conflicting records before continuing.'
price_cols = ['open', 'high', 'low', 'close']
invalid_close = (~np.isfinite(clean['close']) | (clean['close'] <= 0)
                 | (clean['close'] < clean['low']) | (clean['close'] > clean['high']))
clean['close_repaired'] = invalid_close
clean.loc[invalid_close, 'close'] = np.nan
print('Close values needing a repair:', invalid_close.sum())

### Fill within an asset, then recheck the domain
A forward-filled price may fall outside today's range. Here open/high/low are known good because **we injected faults only into close**.
Use the prior close only if it lies in today's range; otherwise use the same day's high/low midpoint as an explicitly marked estimate.
Neither is a recovered observation. In real work, request corrected vendor data first.
Backward fill and interpolation use future values and are unsuitable for features that must be known at that time.

In [ ]:
previous = clean.groupby('ticker')['close'].ffill()
within_range = previous.between(clean['low'], clean['high'])
midpoint = (clean['high'] + clean['low']) / 2
replacement = previous.where(within_range, midpoint)
clean['close'] = clean['close'].fillna(replacement)
invalid_volume = ~np.isfinite(clean['volume']) | (clean['volume'] < 0)
clean['volume_repaired'] = invalid_volume
# Zero here is an explicit missing-volume placeholder, not proof of no trading.
clean.loc[invalid_volume, 'volume'] = 0
clean['return'] = clean.groupby('ticker')['close'].pct_change(fill_method=None)
# A return touches two prices: mark both sides of a repaired close.
clean['return_affected'] = clean['close_repaired'] | clean.groupby('ticker')['close_repaired'].shift(1, fill_value=False)
display(clean.loc[clean['close_repaired'] | clean['volume_repaired']])

### Outlier flags are questions, not instructions to delete
Global Z-score and IQR are simple diagnostics. A large valid move can be the observation a risk report needs most.
Compare thresholds on one asset's returns. Next session adds rolling history and OHLC validation.

In [ ]:
r = clean.loc[clean['ticker'] == 'TECH', 'return']
z = (r - r.mean()) / r.std()
q1, q3 = r.quantile([0.25, 0.75]); iqr = q3 - q1
flags = pd.DataFrame({'z_flag': z.abs() > 3, 'iqr_flag': (r < q1 - 1.5 * iqr) | (r > q3 + 1.5 * iqr)})
display(flags.sum())
# API comparison only: capping changes the distribution; do not silently use it for risk.
display(pd.DataFrame({'observed_return': r, 'capped_example': r.clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)}).head())

## 3. Guided exercise: prove that the repair worked (15 minutes)
Before running the worked checks, write tests for: unique keys, finite positive prices, valid OHLC bounds, and finite nonnegative volume.
Then compare TECH before and after. Explain why “no nulls” alone would miss a forward-fill error.

In [ ]:
# Write your validation checks here

In [ ]:
assert not clean.duplicated(['ticker', 'date']).any()
assert np.isfinite(clean[price_cols]).all().all() and clean[price_cols].gt(0).all().all()
assert clean['high'].ge(clean[['open', 'close']].max(axis=1)).all()
assert clean['low'].le(clean[['open', 'close']].min(axis=1)).all()
assert np.isfinite(clean['volume']).all() and clean['volume'].ge(0).all()
assert clean.groupby('ticker')['return'].apply(lambda s: s.isna().sum()).eq(1).all()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
raw.sort_values('date').plot.scatter(x='date', y='close', ax=axes[0], title='Before: all assets')
for ticker, group in clean.groupby('ticker'):
    axes[1].plot(group['date'], group['close'], label=ticker)
axes[1].set(title='After: repairs flagged', xlabel='Date', ylabel='Price'); axes[1].legend()
plt.show()
clean.to_csv(output_dir / 'clean_prices_demo.csv', index=False)
print('Validation passed; raw data and repair flags are retained.')

## Exit check and next session (5 minutes)
Explain one reason a plausible plot can still hide bad data, and one way a repair changes a statistic.
**October 15:** load data with SQL, build a quality report, and automate a parameterized risk report.
[Homework 2](hw2.md) is still due October 29. Today's tools support its diagnosis and visualization requirements.
Optional practice: compare `dropna`, constant fill, interpolation, percentiles, and string extraction in [cleaning reference](reference/5_data_cleaning_fundamentals.ipynb).